# **💻 Author Information**

**Name:** CHUA JINGXUAN

**Email:** geochua144@gmail.com

**Note:** This Jupyter Notebook was created by CHUA JINGXUAN. If you find any issues, have questions, or want to provide feedback, please don't hesitate to reach out. Thank you for exploring this notebook!

**Date Written**: 03/03/2026 (TUE)

**Last Updated**: 10/03/2026 (TUE)



[![GitHub](https://badges.aleen42.com/src/github.svg)](https://github.com/Kanon14) <a href="https://www.linkedin.com/in/chua-jingxuan-51a300173" target="_blank" style="margin-left: 10px;">
    <img src="https://upload.wikimedia.org/wikipedia/commons/1/19/LinkedIn_logo.svg" alt="LinkedIn Icon" width="80" height="22">
</a>

---

# 🔬 **Experiment for Personal Protective Equipment - Combined Model Computer Vision Model**

# ⚙️ **1.0 Environment Setup**

## 1.1 Configure API Key

To fine-tune RF-DETR, you need to provide your Roboflow API key. Follow these steps:

- Go to your [`Roboflow Settings`](https://app.roboflow.com/settings/api) page. Click `Copy` to copy your private API key.
- In Colab, go to the left pane and click on `Secrets` (🔑).
    - Store your Roboflow API Key under the name `ROBOFLOW_API_KEY`.

In [ ]:
import os
from google.colab import userdata

os.environ["ROBOFLOW_API_KEY"] = userdata.get('ROBOFLOW_API_KEY')

## 1.2 Connecting to GPU for Training Acceleration
Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do that. In case of any problems navigate to `Edit` -> `Notebook settings` -> `Hardware accelerator`, set it to `GPU`, and then click `Save`.

In [ ]:
!nvidia-smi

Wed Mar  4 06:46:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   38C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1.3 Installing Dependencies

In [ ]:
%pip install ultralytics supervision roboflow -q
import ultralytics
ultralytics.checks()

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
Setup complete ✅ (12 CPUs, 53.0 GB RAM, 43.6/235.7 GB disk)


## 1.4 Import Essential Libraries

In [ ]:
import requests
import zipfile
import os
import glob
import cv2
import random
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image as IPyImage, display

# ⬇️ **2.0 Download Personal Protective Equipment Dataset**

YOLO expects the dataset to be in YOLO format. Divide your dataset into three subdirectories: `train`, `valid`, and `test`. Each image should contain its own `.txt` file that holds the annotations for that particular split, along with the corresponding image files. Below is an example of the directory structure:

```
dataset/
├── train/
│   ├── images
│   │   ├── image1.jpg
│   │   └── image2.jpg
|   └── labels
│        ├── label1.txt
│        └── label2.txt
├── valid/
│   ├── images
│   │   ├── image1.jpg
│   │   └── image2.jpg
|   └── labels
│        ├── label1.txt
│        └── label2.txt
└── test/
    ├── images
    │   ├── image1.jpg
    │   └── image2.jpg
    └── labels
        ├── label1.txt
        └── label2.txt
```



In [ ]:
from roboflow import Roboflow
rf = Roboflow()
project = rf.workspace("roboflow-universe-projects").project("personal-protective-equipment-combined-model")
version = project.version(8)
dataset = version.download("yolo26")

In [ ]:
# Display the contents of the YAML that contains the configuration or dataset information
%cat {dataset.location}/data.yaml

# 🛠️ **3.0 Customize YOLO26-small Model Training**

In [ ]:
HOME = os.getcwd()
print(HOME)

<font color="cyan">Estimated Training Time ~ 12 hrs (44k images)</font>

In [ ]:
# Change the current working directory to the HOME directory
%cd {HOME}

# Train a YOLO26-small model for object detection with the specified parameters
from ultralytics import YOLO

model = YOLO("yolo26s.pt")

model.train(
    data = f"{dataset.location}/data.yaml",
    project = output_dir,
    epochs = 50, imgsz=640, patience=10
)

In [ ]:
!ls runs/detect/train

In [ ]:
# Display the confusion matrix image generated during the training process
from IPython.display import Image as IPyImage, display
IPyImage(filename=f'runs/detect/train/confusion_matrix.png', width=600)

In [ ]:
# Display the training results graph, which includes metrics such as loss, precision, recall, and mAP
IPyImage(filename=f'runs/detect/train/results.png', width=600)

# 📊 **4.0 Evaluate Fine-Tuned Model**

Before benchmarking the model, we need to load the best saved checkpoint. To ensure it fits on the GPU, we first need to free up GPU memory. This involves deleting any remaining to previously used objects, triggerting Python's garbage collector, and cleaning the CUDA memory cache.

In [ ]:
import gc
import torch
import weakref

def cleanup_gpu_memory(obj=None, verbose: bool = False):

    if not torch.cuda.is_available():
        if verbose:
            print("[INFO] CUDA is not available. No GPU cleanup needed.")
        return

    def get_memory_stats():
        allocated = torch.cuda.memory_allocated()
        reserved = torch.cuda.memory_reserved()
        return allocated, reserved

    torch.cuda.synchronize()

    if verbose:
        alloc, reserv = get_memory_stats()
        print(f"[Before] Allocated: {alloc / 1024**2:.2f} MB | Reserved: {reserv / 1024**2:.2f} MB")

    # Ensure we drop all strong references
    if obj is not None:
        ref = weakref.ref(obj)
        del obj
        if ref() is not None and verbose:
            print("[WARNING] Object not fully garbage collected yet.")

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

    torch.cuda.synchronize()

    if verbose:
        alloc, reserv = get_memory_stats()
        print(f"[After]  Allocated: {alloc / 1024**2:.2f} MB | Reserved: {reserv / 1024**2:.2f} MB")

In [ ]:
cleanup_gpu_memory(model, verbose=True)

In [ ]:
model = YOLO(model=f"/content/runs/detect/train/weights/best.pt")

In [ ]:
model.predict(
    source = f"{dataset.location}/test/images",
    conf = 0.25,
    project = output_dir,
    name = "predict",
    save = True,
    exist_ok = True
)

In [ ]:
# Plot and visualize images in a 1x1 or 2x2 grid, based on the number of images available
def visualize(result_dir):
    """
    This function accepts a directory path containing images and visualizes them
    - If there are fewer than 4 images, it plots them in a 1x1 grid.
    - If there are 4 or more images, it plots them in a 2x2 grid.
    """

    image_names = glob.glob(os.path.join(result_dir, '*.jpg')) # Get all JPG image file paths from the directory
    if len(image_names) < 4:
        plt.figure(figsize=(10, 7)) # Create a figure for fewer than 4 images
        for i, image_name in enumerate(image_names):
            image = plt.imread(image_name)
            plt.subplot(1, 1, i+1) # Create a 1x1 subplot for each image
            plt.imshow(image)
            plt.axis('off')
            break
    if len(image_names) >= 4:
        plt.figure(figsize=(15, 12)) # Create a figure for 4 or more images
        for i, image_name in enumerate(image_names):
            image = plt.imread(image_name)
            plt.subplot(2, 2, i+1) # Create a 2x2 subplot for the first 4 images
            plt.imshow(image)
            plt.axis('off')
            if i == 3: # Stop after plotting 4 images
                break

    plt.title('Inference Results', fontsize=20) # Set the title of the plot
    plt.tight_layout() # Adjust the layout for better spacing
    plt.show() # Display the plot

In [ ]:
# Call the visualize function to display prediction results
visualize("/content/runs/detect/predict")

# **📂 Compress and Download Folder**
This section zips the folder and provides a downloadable file for local storage or further use. The steps include:

* Compressing the folder into a `.zip` archive.
* Automatically initiating the download of the zipped file.

In [ ]:
import shutil
from google.colab import files

# Define the folder name and output ZIP file name
folder_name = "runs"
zip_file_name = f"{folder_name}.zip"

# Zip the folder
shutil.make_archive(folder_name, 'zip', folder_name)

# Download the zipped folder
files.download(zip_file_name)